# INSTALL DEPENDENCIES

In [1]:
INITAL_SETUP = False

if INITAL_SETUP:
    %pip install fastkaggle
    %pip install kaggle
    %pip install dotenv
    %pip install ipdb


In [2]:
import fastkaggle
import pandas as pd
import numpy as np
from pathlib import Path
import shutil
import os
from fastai.tabular import *
from fastai.tabular.all import *

PATH_LOCAL_PATH_DATA_STORAGE = Path(r'D:\kaggle_data')
COMPETITION_NAME = 'titanic'


# CHECK IF RUNNING ON KAGGLE OR ELSEWHERE

In [3]:
'Kaggle' if fastkaggle.iskaggle else 'Not Kaggle'

'Not Kaggle'

# PREPARE KAGGLE DATA

In [ ]:
if fastkaggle.iskaggle:
    path = os.path.join(Path('../input'), COMPETITION_NAME) # generates path containing competition name
    
else:
    path_temp = Path(COMPETITION_NAME)
    
    # check if data is missing from the desired location
    path_local_competition_data = os.path.join(Path(PATH_LOCAL_PATH_DATA_STORAGE), COMPETITION_NAME) # generates path containing competition name
    if not os.path.isdir(path_local_competition_data):
        fastkaggle.setup_comp('titanic') # download data to temp location
        dest = shutil.move(path_temp, PATH_LOCAL_PATH_DATA_STORAGE) # move the data to the correct location
    
    path = path_local_competition_data # update path to point to folder containing data

print("Data located at: {}".format(str(path)))
    
        

Data located at: D:\kaggle_data\titanic


In [5]:
path_train = os.path.join(path, 'train.csv')

train_df = pd.read_csv(path_train)
train_df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


# Prepare dataset

We will apply the following transformations on the dataset.

* Tokenize the names. For example, "Braund, Mr. Owen Harris" will become ["Braund", "Mr.", "Owen", "Harris"].
* Extract any prefix in the ticket. For example ticket "STON/O2. 3101282" will become "STON/O2." and 3101282.


In [6]:
train_df.isnull().any()

PassengerId    False
Survived       False
Pclass         False
Name           False
Sex            False
Age             True
SibSp          False
Parch          False
Ticket         False
Fare           False
Cabin           True
Embarked        True
dtype: bool

There are n/a values located in
* age
* cabin 
* embarked

We need to clean these up.

In [7]:
def preprocess(df):
    df = df.copy()

    df.replace({'Embarked':(np.nan, '')}, inplace=True)
    df.replace({'Age':(np.nan, '')}, inplace=True)
    df.replace({'Cabin':(np.nan, '')}, inplace=True)
    
    # fill nan with assumed values
    df.fillna({'Embarked': 'S'}, inplace=True)
    df.fillna({'Fare': np.mean(df['Fare'])}, inplace=True)
    age_avg = df['Age'].mean()
    age_std = df['Age'].std()
    df.fillna(
        {'Age': np.random.randint(age_avg - age_std, age_avg + age_std)}, inplace=True)
       
    # Extract the deck letter from the cabin
    df['Deck'] = df['Cabin'].str[0]
    df.fillna({'Deck': ''}, inplace=True)
    
    # hot key for female
    df.replace(
        {'Sex':(['male','female'], [0, 1])}, 
        inplace=True)
    
    df['Embarked'] = df['Embarked'].map( {'S': 0, 'C': 1, 'Q': 2} ).astype('Int64')
    df['family'] = (df['SibSp'] + df['Parch'])
    df['isAlone'] = 1
    df.loc[df['family'] > 0, 'isAlone'] = 0
    delete_columns = ['Name','Ticket','Cabin']
    df.drop(delete_columns, axis=1, inplace=True)
                     
    return df
    
preprocessed_train_df = preprocess(train_df)
preprocessed_train_df.head(5)

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Deck,family,isAlone
0,1,0,3,male,22.0,1,0,7.2500,0,,1,0
1,2,1,1,female,38.0,1,0,71.2833,1,C,1,0
2,3,1,3,female,26.0,0,0,7.9250,0,,0,1
3,4,1,1,female,35.0,1,0,53.1000,0,C,1,0
4,5,0,3,male,35.0,0,0,8.0500,0,,0,1


# Prepare dataloaders

In [ ]:
dep_var = "Survived"
cat_names= ['Pclass', 'Sex', 'Embarked', 'Deck'] # 'isAlone'
cont_names = ['Fare', 'Age', 'SibSp', 'Parch'] # family

dls1 = TabularDataLoaders.from_df(preprocessed_train_df, 
                    cat_names= cat_names,
                    cont_names = cont_names,
                    y_names= dep_var,
                    procs = [Categorify, FillMissing, Normalize]
                                 )


c:\Users\dunca\AppData\Local\Programs\Python\Python314\Lib\site-packages\fastai\torch_core.py:154: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  else as_tensor(x.values, **kwargs) if isinstance(x, (pd.Series, pd.DataFrame))


# Tabular Learner

In [ ]:
learn = tabular_learner(dls1, metrics=accuracy)

# Fit one cycle

In [39]:
learn.fit_one_cycle(n_epoch=10)

epoch,train_loss,valid_loss,accuracy,time
0,0.191201,0.182472,0.612360,00:00
1,0.197776,0.181020,0.612360,00:00
2,0.184124,0.146866,0.612360,00:00
3,0.180226,0.166576,0.612360,00:00
4,0.171466,0.143003,0.612360,00:00
5,0.167054,0.171074,0.612360,00:00
6,0.163901,0.145695,0.612360,00:00
7,0.155778,0.128216,0.612360,00:00
8,0.148956,0.133396,0.612360,00:00
9,0.147435,0.133085,0.612360,00:00


# Quick check of the results

In [11]:
learn.show_results()

,Pclass,Sex,Embarked,Deck,Fare,Age,SibSp,Parch,Survived,Survived_pred
0,2.0,1.0,1.0,1.0,33.0000,29.0,0.0,0.0,1.0,0.655980
1,1.0,1.0,1.0,5.0,26.2833,19.0,0.0,2.0,1.0,0.639588
2,3.0,1.0,1.0,1.0,7.8542,14.0,0.0,0.0,0.0,0.439043
3,3.0,1.0,1.0,8.0,10.4625,2.0,0.0,1.0,0.0,0.899005
4,2.0,2.0,1.0,1.0,10.5000,36.0,0.0,0.0,0.0,-0.071077
5,2.0,2.0,1.0,1.0,0.0000,29.0,0.0,0.0,0.0,-0.126373
6,1.0,2.0,3.0,4.0,90.0000,44.0,2.0,0.0,0.0,0.339375
7,3.0,1.0,1.0,1.0,7.4958,18.0,0.0,0.0,1.0,0.467658
8,2.0,1.0,1.0,1.0,33.0000,29.0,0.0,0.0,1.0,0.655980


# Prepare testing data

Load testing data....

In [12]:
path_test = os.path.join(path, 'test.csv')

test_df = pd.read_csv(path_test)
test_df.head(10)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
5,897,3,"Svensson, Mr. Johan Cervin",male,14.0,0,0,7538,9.2250,NaN,S
6,898,3,"Connolly, Miss. Kate",female,30.0,0,0,330972,7.6292,NaN,Q
7,899,2,"Caldwell, Mr. Albert Francis",male,26.0,1,1,248738,29.0000,NaN,S
8,900,3,"Abrahim, Mrs. Joseph (Sophie Halaut Easu)",female,18.0,0,0,2657,7.2292,NaN,C
9,901,3,"Davies, Mr. John Samuel",male,21.0,2,0,A/4 48871,24.1500,NaN,S


Preprocess testing data...

In [13]:
preprocessed_test_df = preprocess(test_df)
preprocessed_test_df.head(5)

,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Deck,family,isAlone
0,892,3,male,34.5,0,0,7.8292,2,,0,1
1,893,3,female,47.0,1,0,7.0000,0,,1,0
2,894,2,male,62.0,0,0,9.6875,2,,0,1
3,895,3,male,27.0,0,0,8.6625,0,,0,1
4,896,3,female,22.0,1,1,12.2875,0,,2,0


Check to see if any issues are in the testing data (e.g. n/a)

In [14]:
preprocessed_test_df.isnull().any()

PassengerId    False
Pclass         False
Sex            False
Age            False
SibSp          False
Parch          False
Fare           False
Embarked       False
Deck           False
family         False
isAlone        False
dtype: bool

In [16]:
dls_test = learn.dls.test_dl(preprocessed_test_df)

In [29]:
preds, _ = learn.get_preds(dl=dls_test)

In [30]:
preds[:5]

tensor([[-0.1420],
        [ 0.3672],
        [-0.1080],
        [-0.0990],
        [ 0.2189]])

In [31]:
dls_test.show_batch()

,Pclass,Sex,Embarked,Deck,Fare,Age,SibSp,Parch
0,3,male,2,,7.8292,34.5,0.0,0.0
1,3,female,0,,7.0000,47.0,1.0,0.0
2,2,male,2,,9.6875,62.0,0.0,0.0
3,3,male,0,,8.6625,27.0,0.0,0.0
4,3,female,0,,12.2875,22.0,1.0,1.0
5,3,male,0,,9.2250,14.0,0.0,0.0
6,3,female,2,,7.6292,30.0,0.0,0.0
7,2,male,0,,29.0000,26.0,1.0,1.0
8,3,female,1,,7.2292,18.0,0.0,0.0
9,3,male,0,,24.1500,21.0,2.0,0.0


In [ ]:

df_submission = pd.DataFrame({'PassengerId': preprocessed_test_df.PassengerId
                              , 'Survived': preds.flatten()})

In [ ]:
df_submission.head(5)


,PassengerId,Survived
0,892,-0.142005
1,893,0.367194
2,894,-0.107983
3,895,-0.099034
4,896,0.218950


In [38]:
df_submission.to_csv(os.path.join(path, 'submission.csv'), index=False)

# Test data prediction

In [ ]:
tpreds=[]
for i in range(len(preprocessed_test_df)):
    _, _, probs = learn.predict(preprocessed_test_df.iloc[i])
    tpreds+=[probs.numpy()[0]]